# Silver — Tipagem, deduplicação e quarentena

Lê a tabela `poc_b3_modernizacao.bronze.cotacoes` e aplica as regras de qualidade da camada Silver:

1. **Tipagem**: `preco_atual` e `fechamento_anterior` convertidos de string para número.
2. **Deduplicação**: garante uma única linha por combinação `ticker + data_referencia`.
3. **Quarentena**: registros com `data_valida = False` (data de partição divergente do
   conteúdo real, ver ADR-03) ou com preço nulo/negativo são desviados para uma tabela de
   quarentena separada — não quebram o pipeline nem contaminam a tabela principal.

**Entrada:** tabela `poc_b3_modernizacao.bronze.cotacoes`
**Saída:** tabela `poc_b3_modernizacao.silver.cotacoes` (dados limpos e tipados)
         + tabela `poc_b3_modernizacao.silver.cotacoes_quarentena` (registros desviados,
           com motivo da quarentena registrado)

In [0]:
%run ../setup/01_utilitarios_pipeline

In [0]:
# observabilidade - marca inicio da execucao
from datetime import datetime
inicio_execucao = datetime.now()

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# le a tabela bronze
df_bronze = spark.table("poc_b3_modernizacao.bronze.cotacoes")

print(f"Total de registros na Bronze: {df_bronze.count()}")
display(df_bronze)

In [0]:
# tipagem e calculo do motivo de quarentena
df_tipado = (df_bronze
    .withColumn("preco_atual_num", F.col("preco_atual").cast("double"))
    .withColumn("fechamento_anterior_num", F.col("fechamento_anterior").cast("double"))
)

df_tipado = df_tipado.withColumn(
    "motivo_quarentena",
    F.when(F.col("data_valida") == "False", F.lit("data_particao_divergente"))
     .when(F.col("preco_atual_num").isNull() | (F.col("preco_atual_num") <= 0), F.lit("preco_atual_invalido"))
     .when(F.col("fechamento_anterior_num").isNull() | (F.col("fechamento_anterior_num") <= 0), F.lit("fechamento_anterior_invalido"))
     .otherwise(F.lit(None))
)

display(df_tipado)

In [0]:
# separa registros limpos e registros de quarentena
df_limpo = df_tipado.filter(F.col("motivo_quarentena").isNull())
df_quarentena = df_tipado.filter(F.col("motivo_quarentena").isNotNull())

# deduplicacao - garante uma unica linha por ticker + data_referencia
# mantem o registro mais recente, com base em data_carga
df_limpo_dedup = (df_limpo
    .withColumn("_rank", F.row_number().over(
        Window.partitionBy("ticker", "data_referencia").orderBy(F.col("data_carga").desc())
    ))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
)

print(f"Registros limpos (Silver): {df_limpo_dedup.count()}")
print(f"Registros em quarentena: {df_quarentena.count()}")

In [0]:
# schema final limpo - silver.cotacoes
df_silver_final = df_limpo_dedup.select(
    "ticker",
    F.col("data_referencia").cast("date").alias("data_referencia"),
    F.col("preco_atual_num").alias("preco_atual"),
    F.col("fechamento_anterior_num").alias("fechamento_anterior"),
    F.col("data_hora_mercado").cast("timestamp").alias("data_hora_mercado"),
    "data_carga",
    "arquivo_origem",
)

# schema final - silver.cotacoes_quarentena (mantem colunas de auditoria)
df_quarentena_final = df_quarentena.select(
    "ticker",
    F.col("data_referencia").cast("date").alias("data_referencia"),
    F.col("preco_atual_num").alias("preco_atual"),
    F.col("fechamento_anterior_num").alias("fechamento_anterior"),
    "data_valida",
    "motivo_quarentena",
    "data_carga",
    "arquivo_origem",
)

display(df_silver_final)
display(df_quarentena_final)

In [0]:
# grava silver via merge
merge_ou_cria(df_silver_final, "poc_b3_modernizacao.silver.cotacoes", ["ticker", "data_referencia"])
merge_ou_cria(df_quarentena_final, "poc_b3_modernizacao.silver.cotacoes_quarentena", ["ticker", "data_referencia"])

In [0]:
# valida gravacao na silver
print("=== silver.cotacoes (limpa) ===")
display(spark.table("poc_b3_modernizacao.silver.cotacoes").select(
    "ticker", "preco_atual", "fechamento_anterior", "data_referencia"
).orderBy("ticker"))

print("=== silver.cotacoes_quarentena ===")
display(spark.table("poc_b3_modernizacao.silver.cotacoes_quarentena").select(
    "ticker", "data_referencia", "motivo_quarentena"
).orderBy("ticker"))

In [0]:
# verifica schema atual da tabela
spark.table("poc_b3_modernizacao.silver.cotacoes").printSchema()

In [0]:
# observabilidade - registra sucesso da execucao
data_mais_recente_silver = df_silver_final.agg(F.max("data_referencia")).collect()[0][0]

registrar_execucao(
    notebook="03_silver",
    data_referencia=data_mais_recente_silver,
    modo_execucao="reprocessamento_manual",
    status="sucesso",
    inicio=inicio_execucao,
    fim=datetime.now(),
)